In [1]:
!pip install torch -q

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import pickle

print('PyTorch version:', torch.__version__)
print('Device:', 'GPU' if torch.cuda.is_available() else 'CPU')

PyTorch version: 2.11.0+cpu
Device: CPU


In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
df = df.dropna(subset=['text_clean'])
df['text_clean'] = df['text_clean'].astype(str)

print('Data loaded:', df.shape)
print(df['label'].value_counts())

Data loaded: (51055, 4)
label
Normal                  16030
Depression              15085
Suicidal                10634
Anxiety                  3617
Bipolar                  2501
Stress                   2293
Personality disorder      895
Name: count, dtype: int64


In [3]:
# Split data
X = df['text_clean'].values
y = df['label_id'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.50, random_state=42, stratify=y_test)

# Build vocabulary
def build_vocab(texts, max_vocab=20000):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
print('Vocabulary size:', len(vocab))

# Convert text to sequences
def text_to_sequence(texts, vocab, max_len=150):
    sequences = []
    for text in texts:
        seq = [vocab.get(w, 1) for w in text.split()[:max_len]]
        seq += [0] * (max_len - len(seq))
        sequences.append(seq)
    return np.array(sequences)

MAX_LEN = 150
X_train_seq = text_to_sequence(X_train, vocab, MAX_LEN)
X_val_seq = text_to_sequence(X_val, vocab, MAX_LEN)
X_test_seq = text_to_sequence(X_test, vocab, MAX_LEN)

print('Sequences ready! Shape:', X_train_seq.shape)

Vocabulary size: 20000
Sequences ready! Shape: (35738, 150)


In [4]:
# Convert to PyTorch tensors
X_train_tensor = torch.LongTensor(X_train_seq)
X_val_tensor = torch.LongTensor(X_val_seq)
X_test_tensor = torch.LongTensor(X_test_seq)
y_train_tensor = torch.LongTensor(y_train)
y_val_tensor = torch.LongTensor(y_val)
y_test_tensor = torch.LongTensor(y_test)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print('DataLoaders ready!')
print('Train batches:', len(train_loader))

DataLoaders ready!
Train batches: 559


In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.dropout(hidden)
        return self.fc(out)

VOCAB_SIZE = len(vocab)
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = 7

model_lstm = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
print(model_lstm)
print('Parameters:', sum(p.numel() for p in model_lstm.parameters()))

LSTMClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=7, bias=True)
)
Parameters: 3221255


In [6]:
device = torch.device('cpu')
model_lstm = model_lstm.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

EPOCHS = 5

for epoch in range(EPOCHS):
    model_lstm.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model_lstm(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model_lstm.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model_lstm(X_batch.to(device))
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(y_batch.numpy())

    val_acc = accuracy_score(val_true, val_preds)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}')

print('Training done!')

Epoch 1/5 | Loss: 1.0665 | Val Acc: 0.6483
Epoch 2/5 | Loss: 0.7742 | Val Acc: 0.7194
Epoch 3/5 | Loss: 0.6162 | Val Acc: 0.7502
Epoch 4/5 | Loss: 0.5218 | Val Acc: 0.7604
Epoch 5/5 | Loss: 0.4529 | Val Acc: 0.7700
Training done!


In [7]:
# Test evaluation
model_lstm.eval()
test_preds, test_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model_lstm(X_batch.to(device))
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(y_batch.numpy())

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average='weighted')

print('=== MODEL 4: LSTM (PyTorch) ===')
print(f'Accuracy: {test_acc:.4f}')
print(f'F1 Score: {test_f1:.4f}')
print(classification_report(test_true, test_preds))

# Save model
torch.save(model_lstm.state_dict(), '../models/saved/model4_lstm.pt')
pickle.dump(vocab, open('../models/saved/lstm_vocab.pkl', 'wb'))
print('Model 4 saved!')

=== MODEL 4: LSTM (PyTorch) ===
Accuracy: 0.7684
F1 Score: 0.7662
              precision    recall  f1-score   support

           0       0.80      0.82      0.81       543
           1       0.91      0.94      0.92      2405
           2       0.74      0.67      0.71      2263
           3       0.64      0.73      0.68      1595
           4       0.64      0.41      0.50       344
           5       0.74      0.79      0.76       375
           6       0.43      0.47      0.45       134

    accuracy                           0.77      7659
   macro avg       0.70      0.69      0.69      7659
weighted avg       0.77      0.77      0.77      7659

Model 4 saved!
